<a href="https://colab.research.google.com/github/sandeepgangaram/neural-nets/blob/master/makemor/trigram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [4]:
import torch
N = torch.zeros((27,27,27), dtype=torch.int32)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [5]:
for w in words:
  chs = ['.',] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    N[ix1,ix2,ix3] += 1

In [6]:
P = (N+1).float() # +1 for model smoothing to avoid peaks (inf or 0 probability of some combination)
# P.sum(0).shape
P /=P.sum(2, keepdim=True)
P.shape
P[0,:,0].shape


torch.Size([27])

In [7]:
ps = (N[0,:,0] + 1).float()
ps/=ps.sum()
ps.shape


torch.Size([27])

In [8]:
g = torch.Generator().manual_seed(2147483647)
for _ in range(25):
  out = []
  ix1 = 0
  ix2 = torch.multinomial(ps, num_samples=1, replacement=True, generator=g).item()
  out.append(itos[ix2])

  while True:
    p=P[ix1,ix2]
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    if ix == 0:
      break
    out.append(itos[ix])
    ix1 = ix2
    ix2 = ix

  print("".join(out))

ce
za
zogh
uriana
kaydnevonimittain
luwak
ka
da
samiyah
javer
gotai
is
iselivojkwuthda
kaley
maside
en
bvgyn
wynnstihiliven
tahlasuzusfxx
leenlen
iwaisan
win
danne
zam
der


In [9]:
ll = 0
n = 0
for w in words:
  chs = ['.',] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    prob = P[ix1, ix2, ix3]
    ll+=torch.log(prob)
    n+=1

anll = -ll/n
print(anll)

tensor(2.0927)


In [ ]:
# GOAL: maximize likelihood of the data w.r.t. model parameters (statistical modeling)
# equivalent to maximizing the log likelihood (because log is monotonic)
# equivalent to minimizing the negative log likelihood
# equivalent to minimizing the average negative log likelihood

# log(a*b*c) = log(a) + log(b) + log(c)

In [44]:
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    # Flatten the pair (ix1, ix2) into a single index
    xs.append(ix1 * 27 + ix2)
    ys.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print(f"Number of examples: {num}")
print(f"First flattened index: {xs[0].item()}")

Number of examples: 196113
First flattened index: 5


In [45]:
xs.shape
xs.max()

tensor(728)

In [46]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=729).float()
xenc.shape

torch.Size([196113, 729])

In [262]:
xenc.dtype

torch.float32

In [47]:
#summary

# randomly initialize 27 neurons' weights, each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729,27), generator=g)

#forward-pass
xenc = F.one_hot(xs, num_classes=729).float() #input to the networkL one-hot encoding
logits = (xenc@W) #predict log-counts
counts = logits.exp() #counts equivlent to - N (original counts Table from training data)
probs = counts / counts.sum(1, keepdims=True) #probablitites for the next character

#the last two steps together are called "softmax"

In [26]:
import torch

# Example: bigram (1, 5) which might be ('a', 'e')
ix1 = 1
ix2 = 5

# Flatten to a single index
# Since each first character has 27 possible second characters:
flattened_ix = ix1 * 27 + ix2
print(f"Pair ({ix1}, {ix2}) maps to flattened index: {flattened_ix}")

# How to apply this to your whole dataset:
# If xs is shape (N, 2)
xs_pairs = torch.tensor([[0, 1], [1, 2], [26, 26]])
xs_flattened = xs_pairs[:, 0] * 27 + xs_pairs[:, 1]

print("Original pairs:", xs_pairs)
print("Flattened indices:", xs_flattened)
print("Max possible index:", 26 * 27 + 26) # 728

Pair (1, 5) maps to flattened index: 32
Original pairs: tensor([[ 0,  1],
        [ 1,  2],
        [26, 26]])
Flattened indices: tensor([  1,  29, 728])
Max possible index: 728


In [27]:
xs_flattened

tensor([  1,  29, 728])

In [18]:
p
probs.shape

torch.Size([196113, 27])

In [23]:
legal_bigrams = []
for i in range(27):
    for j in range(27):
        ch1 = itos[i]
        ch2 = itos[j]

        # 1. A bigram cannot end in '.' and then expect a 3rd character
        # because '.' is the terminal state.
        if ch2 == '.':
            continue

        # 2. '..' is generally excluded as an input state for names
        if ch1 == '.' and ch2 == '.':
            continue

        legal_bigrams.append((ch1, ch2))

print(f"Total mathematical combinations: 729")
print(f"Structurally 'legal' input bigrams: {len(legal_bigrams)}")
print(f"Eliminated: {729 - len(legal_bigrams)} combinations (mostly 'char + .')")

Total mathematical combinations: 729
Structurally 'legal' input bigrams: 702
Eliminated: 27 combinations (mostly 'char + .')


In [ ]:
# Re-initialize weights for the 729 -> 27 mapping
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [66]:
for i in range(50):
  # forward pass
  # Now xs is 1D (num_examples), so xenc becomes (num_examples, 729)
  xenc = F.one_hot(xs, num_classes=729).float()
  logits = xenc @ W # (N, 729) @ (729, 27) -> (N, 27)
  counts = logits.exp()
  probs = counts / counts.sum(1, keepdims=True)

  # calculate loss (negative log likelihood)
  # probs[torch.arange(num), ys] picks the probability of the correct next character
  loss = -probs[torch.arange(num), ys].log().mean()
  print(f"iteration {i}: loss {loss.item():.4f}")

  # backward pass
  W.grad = None
  loss.backward()

  # update
  W.data -= 100 * W.grad

iteration 0: loss 2.0968
iteration 1: loss 2.0968
iteration 2: loss 2.0967
iteration 3: loss 2.0967
iteration 4: loss 2.0966
iteration 5: loss 2.0966
iteration 6: loss 2.0966
iteration 7: loss 2.0965
iteration 8: loss 2.0965
iteration 9: loss 2.0964
iteration 10: loss 2.0964
iteration 11: loss 2.0963
iteration 12: loss 2.0963
iteration 13: loss 2.0963
iteration 14: loss 2.0962
iteration 15: loss 2.0962
iteration 16: loss 2.0961
iteration 17: loss 2.0961
iteration 18: loss 2.0961
iteration 19: loss 2.0960
iteration 20: loss 2.0960
iteration 21: loss 2.0959
iteration 22: loss 2.0959
iteration 23: loss 2.0959
iteration 24: loss 2.0958
iteration 25: loss 2.0958
iteration 26: loss 2.0958
iteration 27: loss 2.0957
iteration 28: loss 2.0957
iteration 29: loss 2.0956
iteration 30: loss 2.0956
iteration 31: loss 2.0956
iteration 32: loss 2.0955
iteration 33: loss 2.0955
iteration 34: loss 2.0955
iteration 35: loss 2.0954
iteration 36: loss 2.0954
iteration 37: loss 2.0953
iteration 38: loss 2.0

In [67]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
  out = []
  ix1 = 0 # '.'
  # We need the first character after '.'
  # To get this from our model, we use the bigram index for ('.', '.')
  # Or we can sample the first character independently if favored,
  # but let's stick to the trained weights
  ix2 = 0 # initially '.'

  while True:
    # Flatten the current bigram (ix1, ix2) to index the weights
    x_flat = ix1 * 27 + ix2
    xenc = F.one_hot(torch.tensor([x_flat]), num_classes=729).float()
    logits = xenc @ W
    counts = logits.exp()
    p = counts / counts.sum(1, keepdims=True)

    ix_next = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    if ix_next == 0: # end token
      break

    out.append(itos[ix_next])
    # Shift the window
    ix1 = ix2
    ix2 = ix_next

  print("".join(out))

zexza
zogjkuriana
otah
oll
imittain
luwan
ka
or
zamiyah
pathrighton


In [ ]:
zexzdfzjglkuriana
otxhkmvlzimjtna
orakayk
ka
oa
zamiyaubjtbhrightqp
is
isqckvugkwpteda
kaley
zaside

In [ ]:
zexzdfzjglkurxycezkwyhhmvlzimjtna
orlkfdkzka
og
zw
zvpbbpwkhrggitmj
idzjedktvojkw
t
dagkkjeykazsidszhtkavnrnfrqtbspmhwcjdewvtahlvsuqysfxxblgjxlhgfiwuipgna
opfdnipkezktsdesu
zixht

In [ ]:
cexze.
momasurailezitynn.
konimittain.
llayn.
ka.